# 07 · Abstraction

**Goal:** learn how to define a common interface that subclasses *must* implement, using
Python's `abc` module — hiding "how" behind a clean "what".

### What is abstraction?

Abstraction means exposing only the **essential** details of an object, and hiding the
**implementation complexity** behind a simple interface.

When you drive a car, you use the steering wheel and pedals (the interface) without needing to
know how the engine's combustion actually works (the implementation). That's abstraction.

In OOP, we often express this by defining an **abstract base class**: a class that defines
*what* methods subclasses must have, without saying *how* they work.

### Without abstraction: nothing enforces the "contract"

If you just write a plain base class with a method that does nothing, there's no guarantee a
subclass actually implements it — you might forget, and only find out at runtime, potentially
far from where the bug actually is.

In [1]:
class Shape:
    def area(self):
        pass   # nothing stops a subclass from just... not overriding this

class Triangle(Shape):
    pass  # oops, forgot to implement area()

t = Triangle()
print(t.area())   # returns None silently -- a bug that's easy to miss!

None


### With abstraction: `abc.ABC` and `@abstractmethod`

The `abc` module (Abstract Base Classes) lets you **enforce** that subclasses implement
certain methods. Trying to instantiate a class that hasn't implemented all abstract methods
raises an error immediately — catching the bug at the earliest possible point.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):              # inherit from ABC to make this an abstract base class
    @abstractmethod
    def area(self):
        """Subclasses MUST implement this."""
        pass

    @abstractmethod
    def perimeter(self):
        """Subclasses MUST implement this too."""
        pass

    def describe(self):
        # Regular (non-abstract) methods are allowed too, and are inherited normally
        return f"This shape has area {self.area()} and perimeter {self.perimeter()}"


try:
    s = Shape()   # can't instantiate an abstract class directly!
except TypeError as e:
    print("Error:", e)

In [ ]:
class Rectangle(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h

    def area(self):
        return self.w * self.h

    def perimeter(self):
        return 2 * (self.w + self.h)


r = Rectangle(4, 5)
print(r.describe())   # inherited concrete method, uses the abstract methods internally

In [ ]:
class IncompleteShape(Shape):
    def area(self):
        return 0
    # forgot to implement perimeter()!

try:
    incomplete = IncompleteShape()
except TypeError as e:
    print("Error:", e)   # Python catches the missing implementation immediately

### Abstraction vs. Encapsulation — don't confuse them

These two pillars are related but different:

| | Encapsulation | Abstraction |
|---|---|---|
| Focus | Protecting/controlling **data** | Hiding **implementation complexity** |
| How (in Python) | `_protected`, `__private`, `@property` | `ABC`, `@abstractmethod` |
| Answers | "How is the data accessed/changed?" | "What can this object do, without saying how?" |

### Real-world style example: a payment system

This is the kind of pattern abstraction is *for*: define a common interface so the rest of
your code doesn't care which specific payment method is being used.

In [ ]:
from abc import ABC, abstractmethod

class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount):
        pass

class CreditCard(PaymentMethod):
    def pay(self, amount):
        return f"Paid {amount} using Credit Card"

class UPI(PaymentMethod):
    def pay(self, amount):
        return f"Paid {amount} using UPI"

def checkout(payment_method: PaymentMethod, amount):
    # This function doesn't care HOW payment happens, only that .pay() exists
    print(payment_method.pay(amount))

checkout(CreditCard(), 500)
checkout(UPI(), 250)

# Adding a new payment method (e.g. PayPal) later requires ZERO changes to checkout()

### 🧠 Quick check

1. What happens if you try to instantiate a class that inherits from `ABC` but hasn't
   implemented all its `@abstractmethod`s?
2. What's the key difference between abstraction and encapsulation?
3. Can an abstract base class have regular (non-abstract) methods too?

<details>
<summary>Answers</summary>

1. Python raises a `TypeError` at instantiation time, listing the missing method(s).
2. Encapsulation controls access to *data*; abstraction hides *implementation details* behind
   a simple, enforced interface.
3. Yes — abstract classes can mix abstract methods (must be overridden) with regular concrete
   methods (inherited as-is).
</details>

### ✍️ Practice

1. Create an abstract class `Vehicle` with abstract methods `start_engine()` and `stop_engine()`.
2. Implement `Car` and `Motorcycle` subclasses.
3. Try instantiating `Vehicle` directly and confirm you get a `TypeError`.
4. Add a concrete method `service()` on `Vehicle` that all subclasses inherit for free.

Continue to **`08_magic_dunder_methods.ipynb`** next.